# 01 - Decay probability in 1D

This notebook studies the simplest phenomenological question for a long-lived particle (LLP): what is the probability that it decays inside a given distance interval along its trajectory?

The goal is not to model a real detector. The regions below are toy geometries with distance scales chosen to make the physics transparent.

## Lab-frame decay length

For a particle with proper decay length `ctau` and boost `beta gamma`, the mean decay length in the lab frame is

$$
L_{\rm lab} = \beta\gamma\,c\tau.
$$

In this project, `ctau` is entered directly in meters and `beta gamma` is dimensionless.

## Exponential decay law

The probability density for decay after a path length $L$ is

$$
f(L) = \frac{1}{L_{\rm lab}}\exp\left(-\frac{L}{L_{\rm lab}}\right),
$$

and the survival probability is

$$
S(L) = \exp\left(-\frac{L}{L_{\rm lab}}\right).
$$

Therefore the probability to decay between two distances $L_1$ and $L_2$ is

$$
P(L_1 < L < L_2) = S(L_1) - S(L_2).
$$

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))
FIGURES = PROJECT_ROOT / "figures"
FIGURES.mkdir(exist_ok=True)

from llp_acceptance.decay import decay_pdf, survival_probability
from llp_acceptance.geometry import TOY_REGIONS_1D, probability_in_region
from llp_acceptance.kinematics import lab_decay_length
from llp_acceptance.plotting import (
    plot_decay_pdf,
    plot_probability_vs_ctau,
    plot_survival_probability,
)


## Toy detector regions

The project defines simple one-dimensional regions along the LLP path. They are not real detector dimensions.

In [ ]:
for region in TOY_REGIONS_1D:
    print(f"{region.name:18s}: {region.L_min:g} m < L < {region.L_max:g} m")


## Probability as a function of ctau

For fixed boost, scanning `ctau` scans the lab-frame decay length. Each toy region peaks when the typical decay length is comparable to that region's distance scale.

In [ ]:
ctau_values = np.logspace(-3, 4, 500)  # meters
betagamma = 10.0

fig, ax = plot_probability_vs_ctau(
    TOY_REGIONS_1D,
    ctau_values,
    betagamma=betagamma,
    output_path=FIGURES / "probability_vs_ctau_regions_1d.png",
)
plt.show()


A very short-lived LLP decays before reaching displaced regions. A very long-lived LLP often escapes the region before decaying. The probability is largest in the intermediate regime where `L_lab` is close to the geometric scale of the region.

In [ ]:
for region in TOY_REGIONS_1D:
    probabilities = probability_in_region(region, lab_decay_length(ctau_values, betagamma))
    index = int(np.argmax(probabilities))
    print(
        f"{region.name:18s}: max P = {probabilities[index]:.3f} "
        f"at ctau = {ctau_values[index]:.3g} m"
    )


## Survival probability and spatial PDF

The same exponential law can be visualized through the survival probability and the decay-position density.

In [ ]:
L_values = np.linspace(0.0, 30.0, 500)
L_lab_example = 10.0  # meters

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
plot_survival_probability(L_values, L_lab_example, ax=axes[0])
plot_decay_pdf(L_values, L_lab_example, ax=axes[1])
fig.tight_layout()
fig.savefig(FIGURES / "survival_and_decay_pdf_1d.png", bbox_inches="tight", dpi=150)
plt.show()


## Interpretation

- If `ctau` is too small, the LLP decays promptly or before reaching a displaced toy region.
- If `ctau` is too large, the LLP is likely to pass through the region without decaying.
- If `beta gamma ctau` is comparable to the region's distance scale, the decay probability inside that region is appreciable.
- A larger boost moves the same proper lifetime to larger lab-frame distances.